<a href="https://colab.research.google.com/github/dhaya-2007/AI/blob/main/unit_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:

import pandas as pd
import re
df = pd.read_csv("dataset.csv")

print("First five rows:")
print(df.head())
print("\nDataset shape:")
print(df.shape)
print("\nMissing values:")
print(df.isnull().sum())
df = df.dropna(subset=["Text", "language"])
df = df.drop_duplicates(subset=["Text", "language"])
df["Text"] = df["Text"].astype(str).str.strip()
df = df[df["Text"] != ""]
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["Text"] = df["Text"].apply(preprocess_text)

print("\nCleaned dataset:")
print(df.head())

print("\nFinal dataset shape:")
print(df.shape)

print("\nLanguage distribution:")
print(df["language"].value_counts())


First five rows:
                                                Text  language
0  klement gottwaldi surnukeha palsameeriti ning ...  Estonian
1  sebes joseph pereira thomas  på eng the jesuit...   Swedish
2  ถนนเจริญกรุง อักษรโรมัน thanon charoen krung เ...      Thai
3  விசாகப்பட்டினம் தமிழ்ச்சங்கத்தை இந்துப் பத்திர...     Tamil
4  de spons behoort tot het geslacht haliclona en...     Dutch

Dataset shape:
(22684, 2)

Missing values:
Text        0
language    0
dtype: int64

Cleaned dataset:
                                                Text  language
0  klement gottwaldi surnukeha palsameeriti ning ...  Estonian
1  sebes joseph pereira thomas på eng the jesuits...   Swedish
2  ถนนเจริญกรุง อักษรโรมัน thanon charoen krung เ...      Thai
3  விசாகப்பட்டினம் தமிழ்ச்சங்கத்தை இந்துப் பத்திர...     Tamil
4  de spons behoort tot het geslacht haliclona en...     Dutch

Final dataset shape:
(21859, 2)

Language distribution:
language
Turkish       1001
Japanese      1000
Korean        1000
C

In [3]:

import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
df = pd.read_csv("dataset.csv")
df = df.dropna(subset=["Text", "language"])
df["Text"] = df["Text"].astype(str).str.strip()
df = df[df["Text"] != ""]
df = df.drop_duplicates(subset=["Text", "language"])
df = (
    df.groupby("language", group_keys=False)
      .sample(n=500, random_state=42)
      .reset_index(drop=True)
)

texts = df["Text"].tolist()
tokenizer = Tokenizer(
    num_words=20000,
    oov_token="<OOV>",
    lower=False,
    filters=""
)

tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
MAX_LEN = 150

padded_sequences = pad_sequences(
    sequences,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

print("Original text:")
print(texts[0])

print("\nNumerical sequence:")
print(sequences[0])

print("\nPadded sequence:")
print(padded_sequences[0])

print("\nPadded data shape:")
print(padded_sequences.shape)

print("\nVocabulary size:")
print(len(tokenizer.word_index))


Original text:
يبحث الناقد في المذاهب التقليدية عن قصدية الفنان أي ما أراد أن يعرضه وموقفه من الحياة ورؤيته لها من منطلق أن يعبر عن الواقع وعن اتجاهات المجتمع أما في " النقد البنيوى " فالقصدية ليست قصدية الفنان وانما قصدية البناء الشكلى إذ أن الشكل هو الذي يقصد أما الفنان فهو غائب فلا جدوى من بحث الناقد في أى شيء ميتافيزيقى أما في نقد" ما بعد الحداثة "فإن المتلقى هو الذي يحدد "القصدية " ان وجدت إذ ان وجود القصدية يتوقف على تصور المتلقى للشكل و على تفاعله معه وهكذا لا يقوم النقد على دراسة بناء الشكل للعمل الفنى وانما على الرغبات و الأهواء التي تنطلق من المتلقى وكان النقد في أعمال الفن التي أبدعت بمفهوم البناء التوليفى في الفن الحديث يقوم بمهمة عرض قضية اقحام الأشكال على اللوحة إذ أن العمل الفنى في اطار قصد معين وبالنسبة للنقد البنيوى فإنه يقر بغياب الفنان منذ البداية وبالتالى يتجاهل قصديته ولا يربط دلالة الأشكال في العمل الفنى رجوعا إلى الطبيعة وإنما يحصل على مدلولها من خلال علاقتها بالأ شكال الأخرى في إطار التركيب البنائى كله وهكذا استبدل مفهوم المحاكاة بفكرة "المونتاج"

Numerical sequ

In [4]:

import pandas as pd
import re
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
df = pd.read_csv("dataset.csv")
df = df.dropna(subset=["Text", "language"])
df["Text"] = df["Text"].astype(str).str.strip()
df = df[df["Text"] != ""]
df = df.drop_duplicates(subset=["Text", "language"])
df = (
    df.groupby("language", group_keys=False)
      .sample(n=500, random_state=42)
      .reset_index(drop=True)
)
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["Text"] = df["Text"].apply(preprocess_text)
encoder = LabelEncoder()
y = encoder.fit_transform(df["language"])

num_classes = len(encoder.classes_)
MAX_WORDS = 20000
MAX_LEN = 150

tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<OOV>",
    lower=False,
    filters=""
)

tokenizer.fit_on_texts(df["Text"])

sequences = tokenizer.texts_to_sequences(df["Text"])

X = pad_sequences(
    sequences,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
model = Sequential([
    Embedding(
        input_dim=MAX_WORDS,
        output_dim=64,
        mask_zero=True
    ),
    LSTM(64),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dropout(0.2),
    Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

loss, accuracy = model.evaluate(X_test, y_test, verbose=0)

print("\nTest Loss:", loss)
print("Test Accuracy:", accuracy)
model.save("language_lstm_model.keras")

print("\nModel training completed.")


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
124/124 ━━━━━━━━━━━━━━━━━━━━ 24s 166ms/step - accuracy: 0.1294 - loss: 2.7623 - val_accuracy: 0.2170 - val_loss: 2.3281
Epoch 2/10
124/124 ━━━━━━━━━━━━━━━━━━━━ 42s 175ms/step - accuracy: 0.3282 - loss: 2.0286 - val_accuracy: 0.5193 - val_loss: 1.5745
Epoch 3/10
124/124 ━━━━━━━━━━━━━━━━━━━━ 41s 176ms/step - accuracy: 0.5357 - loss: 1.3907 - val_accuracy: 0.6852 - val_loss: 1.0283
Epoch 4/10
124/124 ━━━━━━━━━━━━━━━━━━━━ 39s 158ms/step - accuracy: 0.7360 - loss: 0.8599 - val_accuracy: 0.8693 - val_loss: 0.5892
Epoch 5/10
124/124 ━━━━━━━━━━━━━━━━━━━━ 20s 160ms/step - accuracy: 0.8452 - loss: 0.5466 - val_accuracy: 0.8830 - val_loss: 0.4733
Epoch 6/10
124/124 ━━━━━━━━━━━━━━━━━━━━ 20s 159ms/step - accuracy: 0.8689 - loss: 0.4836 - val_accuracy: 0.9125 - val_loss: 0.3439
Epoch 7/10
124/124 ━━━━━━━━━━━━━━━━━━━━ 20s 159ms/step - accuracy: 0.9119 - loss: 0.3014 - val_accuracy: 0.9148 - val_loss: 0.2874
Epoch 8/10
124/124 ━━━━━━━━━━━━━━━━━━━━ 21s 166ms/step - accuracy: 0.9114 - loss: 0

4 Train and evaluate the model using accuracy,
precision, recall and F1-score.

In [5]:

import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
y_prob = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

print("========== EVALUATION RESULTS ==========")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        labels=np.arange(len(encoder.classes_)),
        target_names=encoder.classes_,
        zero_division=0
    )
)


cm = confusion_matrix(
    y_test,
    y_pred,
    labels=np.arange(len(encoder.classes_))
)

print("\nConfusion Matrix:")
print(cm)


========== EVALUATION RESULTS ==========
Accuracy  : 0.8845
Precision : 0.8924
Recall    : 0.8845
F1-score  : 0.8769

Classification Report:
              precision    recall  f1-score   support

      Arabic       1.00      0.64      0.78       100
     Chinese       0.49      0.88      0.63       100
       Dutch       1.00      0.96      0.98       100
     English       0.88      0.96      0.92       100
    Estonian       0.89      0.55      0.68       100
      French       1.00      0.99      0.99       100
       Hindi       0.99      1.00      1.00       100
  Indonesian       0.96      0.99      0.98       100
    Japanese       0.45      0.14      0.21       100
      Korean       0.77      0.99      0.86       100
       Latin       0.94      0.90      0.92       100
     Persian       0.99      0.95      0.97       100
   Portugese       0.98      0.99      0.99       100
      Pushto       0.96      0.95      0.95       100
    Romanian       0.97      0.97      0.97     

5.      Display the confusion matrix and predict the
language of five unseen text samples.

In [17]:
!pip install langid

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 29.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langid: filename=langid-1.1.6-py3-none-any.whl size=1941216 sha256=164311d98354da2afbaeadf387b68cbb1cad95d0605e3113f02ecedb79a7f72d
  Stored in directory: /root/.cache/pip/wheels/50/d7/8b/f20e951da531c61b96b87311b48dd1cc6fedaf1e37c581aaee
Successfully built langid


In [19]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import langid

text_input = widgets.Textarea(
    placeholder="Enter a sentence in any language...",
    description="Text:",
    layout=widgets.Layout(width="80%", height="100px")
)

button = widgets.Button(
    description="Identify Language",
    button_style="success",
    icon="search"
)

output = widgets.Output()

def predict_realtime(sender):
    with output:
        clear_output(wait=True)

        text = text_input.value.strip()

        if not text:
            print("Please enter some text.")
            return

        if len(text) < 3:
            print("Please enter a longer sentence.")
            return

        try:
            language_code, score = langid.classify(text)

            language_names = {
                "en": "English",
                "ta": "Tamil",
                "hi": "Hindi",
                "te": "Telugu",
                "kn": "Kannada",
                "ml": "Malayalam",
                "bn": "Bengali",
                "gu": "Gujarati",
                "pa": "Punjabi",
                "mr": "Marathi",
                "ur": "Urdu",
                "ar": "Arabic",
                "zh": "Chinese",
                "ja": "Japanese",
                "ko": "Korean",
                "ru": "Russian",
                "th": "Thai",
                "fr": "French",
                "de": "German",
                "es": "Spanish"
            }

            language = language_names.get(
                language_code, language_code
            )

            print("========== LANGUAGE DETECTION ==========")
            print("Input Text :", text)
            print("Language   :", language)
            print("Code       :", language_code)
            print("Model Score:", score)

        except Exception as e:
            print("Error:", e)

button.on_click(predict_realtime)

display(text_input, button, output)

Textarea(value='', description='Text:', layout=Layout(height='100px', width='80%'), placeholder='Enter a sente…

Button(button_style='success', description='Identify Language', icon='search', style=ButtonStyle())

Output()